# 使用 Chroma 和 OpenAI 实现稳健的问答 

本 notebook 将逐步指导你如何针对一组数据回答问题，使用的是开源嵌入数据库 [Chroma](https://trychroma.com)，以及 OpenAI 的[文本嵌入](https://platform.openai.com/docs/guides/embeddings/use-cases)和 [chat completion](https://platform.openai.com/docs/guides/chat) API。 

此外，本 notebook 还展示了让问答系统更稳健时需要做出的一些权衡。我们将会看到，*简单的查询并不总能得到最好的结果*！ 

## 使用 LLM 进行问答

像 OpenAI 的 ChatGPT 这样的大语言模型（LLM）可以用来回答关于模型可能未曾训练过或无法访问的数据的问题。例如：

- 电子邮件和笔记等个人数据
- 档案或法律文件等高度专业化的数据
- 近期新闻报道等新产生的数据

为了克服这一限制，我们可以使用一种像 LLM 本身一样适合用自然语言查询的数据存储。像 Chroma 这样的嵌入存储会将文档表示为[嵌入](https://openai.com/blog/introducing-text-and-code-embeddings)，并与文档本身一起存储。 

通过嵌入文本查询，Chroma 可以找到相关文档，然后我们把这些文档传给 LLM 来回答问题。我们将展示这种方法的详细示例和变体。 

# 设置与准备

首先确保已安装所需的 python 依赖。 

In [1]:
%pip install -qU openai chromadb pandas

Note: you may need to restart the kernel to use updated packages.


本 notebook 全程使用 OpenAI 的 API。你可以从 [https://beta.openai.com/account/api-keys](https://beta.openai.com/account/api-keys) 获取 API 密钥。

你可以在终端中执行命令 `export OPENAI_API_KEY=sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx`，将 API 密钥添加为环境变量。注意，如果之前尚未设置该环境变量，你需要重新加载 notebook。或者，你也可以在 notebook 中设置，见下文。 

In [2]:
import os
from openai import OpenAI

# Uncomment the following line to set the environment variable in the notebook
# os.environ["OPENAI_API_KEY"] = 'sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    client = OpenAI(api_key=api_key)
    print("OpenAI client is ready")
else:
    print("OPENAI_API_KEY environment variable not found")

OpenAI client is ready


In [3]:
# Set the model for all API calls
OPENAI_MODEL = "gpt-4o"

# 数据集

本 notebook 全程使用 [SciFact 数据集](https://github.com/allenai/scifact)。这是一个经过整理的、由专家标注的科学论断数据集，并附有一个由论文标题和摘要组成的文本语料库。根据语料库中的文档，每个论断可能得到支持、被反驳，或者正反两方面都没有足够证据。 

有了作为真实标签的语料库，我们就能考察以下几种 LLM 问答方法的表现如何。 

In [4]:
# Load the claim dataset
import pandas as pd

data_path = '../../data'

claim_df = pd.read_json(f'{data_path}/scifact_claims.jsonl', lines=True)
claim_df.head()

,id,claim,evidence,cited_doc_ids
0,1,0-dimensional biomaterials show inductive prop...,{},[31715818]
1,3,"1,000 genomes project enables mapping of genet...","{'14717500': [{'sentences': [2, 5], 'label': '...",[14717500]
2,5,1/2000 in UK have abnormal PrP positivity.,"{'13734012': [{'sentences': [4], 'label': 'SUP...",[13734012]
3,13,5% of perinatal mortality is due to low birth ...,{},[1606628]
4,36,A deficiency of vitamin B12 increases blood le...,{},"[5152028, 11705328]"


# 直接询问模型

ChatGPT 在大量科学信息上进行过训练。作为基线，我们希望了解模型在没有任何额外上下文的情况下已经知道什么。这将帮助我们校准整体表现。 

我们构造一个合适的提示，附上一些示例事实，然后用数据集中的每个论断查询模型。我们要求模型将论断评估为 'True'、'False'，或者在正反两方面都没有足够证据时评估为 'NEE'。 

In [5]:
def build_prompt(claim):
    return [
        {"role": "system", "content": "I will ask you to assess a scientific claim. Output only the text 'True' if the claim is true, 'False' if the claim is false, or 'NEE' if there's not enough evidence."},
        {"role": "user", "content": f"""
Example:

Claim:
0-dimensional biomaterials show inductive properties.

Assessment:
False

Claim:
1/2000 in UK have abnormal PrP positivity.

Assessment:
True

Claim:
Aspirin inhibits the production of PGE2.

Assessment:
False

End of examples. Assess the following claim:

Claim:
{claim}

Assessment:
"""}
    ]


def assess_claims(claims):
    responses = []
    # Query the OpenAI API
    for claim in claims:
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=build_prompt(claim),
            max_tokens=3,
        )
        # Strip any punctuation or whitespace from the response
        responses.append(response.choices[0].message.content.strip('., '))

    return responses

我们从数据集中抽取 50 个论断

In [6]:
# Let's take a look at 50 claims
samples = claim_df.sample(50)

claims = samples['claim'].tolist()


我们根据数据集确定真实标签。根据数据集的描述，每个论断要么得到证据支持，要么被证据反驳，要么正反两方面都没有足够证据。 

In [7]:
def get_groundtruth(evidence):
    groundtruth = []
    for e in evidence:
        # Evidence is empty
        if len(e) == 0:
            groundtruth.append('NEE')
        else:
            # In this dataset, all evidence for a given claim is consistent, either SUPPORT or CONTRADICT
            if list(e.values())[0][0]['label'] == 'SUPPORT':
                groundtruth.append('True')
            else:
                groundtruth.append('False')
    return groundtruth

In [8]:
evidence = samples['evidence'].tolist()
groundtruth = get_groundtruth(evidence)

我们还会输出混淆矩阵，以易于阅读的表格形式比较模型的评估与真实标签。 

In [9]:
def confusion_matrix(inferred, groundtruth):
    assert len(inferred) == len(groundtruth)
    confusion = {
        'True': {'True': 0, 'False': 0, 'NEE': 0},
        'False': {'True': 0, 'False': 0, 'NEE': 0},
        'NEE': {'True': 0, 'False': 0, 'NEE': 0},
    }
    for i, g in zip(inferred, groundtruth):
        confusion[i][g] += 1

    # Pretty print the confusion matrix
    print('\tGroundtruth')
    print('\tTrue\tFalse\tNEE')
    for i in confusion:
        print(i, end='\t')
        for g in confusion[i]:
            print(confusion[i][g], end='\t')
        print()

    return confusion

我们要求模型在没有额外上下文的情况下直接评估这些论断。 

In [10]:
gpt_inferred = assess_claims(claims)
confusion_matrix(gpt_inferred, groundtruth)

	Groundtruth
	True	False	NEE
True	9	3	15	
False	0	3	2	
NEE	8	6	4	


{'True': {'True': 9, 'False': 3, 'NEE': 15},
 'False': {'True': 0, 'False': 3, 'NEE': 2},
 'NEE': {'True': 8, 'False': 6, 'NEE': 4}}

## 结果

从这些结果可以看出，LLM 强烈倾向于将论断评估为真，即使它们是假的；同时也倾向于将错误的论断评估为证据不足。注意，这里的"证据不足"是相对于模型在没有额外上下文、孤立地评估论断而言的。


# 添加上下文 

现在我们加入由论文标题和摘要组成的语料库所提供的额外上下文。本节展示如何使用 OpenAI 文本嵌入将文本语料库加载到 Chroma 中。 

首先，加载文本语料库。 

In [11]:
# Load the corpus into a dataframe
corpus_df = pd.read_json(f'{data_path}/scifact_corpus.jsonl', lines=True)
corpus_df.head()

,doc_id,title,abstract,structured
0,4983,Microstructural development of human newborn c...,[Alterations of the architecture of cerebral w...,False
1,5836,Induction of myelodysplasia by myeloid-derived...,[Myelodysplastic syndromes (MDS) are age-depen...,False
2,7912,"BC1 RNA, the transcript from a master gene for...",[ID elements are short interspersed elements (...,False
3,18670,The DNA Methylome of Human Peripheral Blood Mo...,[DNA methylation plays an important role in bi...,False
4,19238,The human myelin basic protein gene is include...,[Two human Golli (for gene expressed in the ol...,False


## 将语料库加载到 Chroma

下一步是将语料库加载到 Chroma 中。只要提供嵌入函数，Chroma 就会自动处理每个文档的嵌入，并将其与文本和元数据一起存储，使查询变得简单。

我们实例化一个（临时的）Chroma 客户端，并为 SciFact 标题和摘要语料库创建一个集合。 
Chroma 也可以以持久化配置实例化；详情请参阅 [Chroma 文档](https://docs.trychroma.com/usage-guide?lang=py)。 

In [12]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

# We initialize an embedding function, and provide it to the collection.
embedding_function = OpenAIEmbeddingFunction(api_key=os.getenv("OPENAI_API_KEY"))

chroma_client = chromadb.Client() # Ephemeral by default
scifact_corpus_collection = chroma_client.create_collection(name='scifact_corpus', embedding_function=embedding_function)

接下来将语料库加载到 Chroma。由于数据加载非常占用内存，我们建议采用分批加载的方式，每批 50-1000 条。在本示例中，加载整个语料库只需一分多钟。嵌入会使用我们之前指定的 `embedding_function` 在后台自动完成。

In [13]:
batch_size = 100

for i in range(0, len(corpus_df), batch_size):
    batch_df = corpus_df[i:i+batch_size]
    scifact_corpus_collection.add(
        ids=batch_df['doc_id'].apply(lambda x: str(x)).tolist(), # Chroma takes string IDs.
        documents=(batch_df['title'] + '. ' + batch_df['abstract'].apply(lambda x: ' '.join(x))).to_list(), # We concatenate the title and abstract.
        metadatas=[{"structured": structured} for structured in batch_df['structured'].to_list()] # We also store the metadata, though we don't use it in this example.
    )

## 检索上下文

接下来，我们从语料库中检索可能与样本中每个论断相关的文档，并将其作为上下文提供给 LLM 来评估论断。我们根据嵌入距离为每个论断检索最相关的 3 篇文档。 

In [14]:
claim_query_result = scifact_corpus_collection.query(query_texts=claims, include=['documents', 'distances'], n_results=3)

我们创建一个新的提示，这一次会考虑从语料库中检索到的额外上下文。 

In [15]:
def build_prompt_with_context(claim, context):
    return [{'role': 'system', 'content': "I will ask you to assess whether a particular scientific claim, based on evidence provided. Output only the text 'True' if the claim is true, 'False' if the claim is false, or 'NEE' if there's not enough evidence."},
            {'role': 'user', 'content': f""""
The evidence is the following:

{' '.join(context)}

Assess the following claim on the basis of the evidence. Output only the text 'True' if the claim is true, 'False' if the claim is false, or 'NEE' if there's not enough evidence. Do not output any other text.

Claim:
{claim}

Assessment:
"""}]


def assess_claims_with_context(claims, contexts):
    responses = []
    # Query the OpenAI API
    for claim, context in zip(claims, contexts):
        # If no evidence is provided, return NEE
        if len(context) == 0:
            responses.append('NEE')
            continue
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=build_prompt_with_context(claim=claim, context=context),
            max_tokens=3,
        )
        # Strip any punctuation or whitespace from the response
        responses.append(response.choices[0].message.content.strip('., '))

    return responses

然后让模型结合检索到的上下文评估这些论断。 

In [16]:
gpt_with_context_evaluation = assess_claims_with_context(claims, claim_query_result['documents'])
confusion_matrix(gpt_with_context_evaluation, groundtruth)

	Groundtruth
	True	False	NEE
True	13	1	4	
False	1	10	2	
NEE	3	1	15	


{'True': {'True': 13, 'False': 1, 'NEE': 4},
 'False': {'True': 1, 'False': 10, 'NEE': 2},
 'NEE': {'True': 3, 'False': 1, 'NEE': 15}}

## 结果

可以看到，模型的整体表现更好了，并且在正确识别错误论断方面有了显著提升。此外，大多数 NEE 的情况现在也能被正确识别。

查看检索到的文档会发现，它们有时与论断并不相关——这会使模型被额外信息干扰，即使信息不相关，模型也可能认为存在足够的证据。之所以出现这种情况，是因为我们总是请求 3 篇"最"相关的文档，但超过某个程度后，这些文档可能根本不相关。  

## 按相关性过滤上下文

除了文档本身，Chroma 还会返回一个距离分数。我们可以尝试对距离设置阈值，让更少的不相关文档进入我们提供给模型的上下文。 

如果按阈值过滤后没有剩下任何上下文文档，我们就跳过模型，直接返回证据不足。 

In [17]:
def filter_query_result(query_result, distance_threshold=0.25):
# For each query result, retain only the documents whose distance is below the threshold
    for ids, docs, distances in zip(query_result['ids'], query_result['documents'], query_result['distances']):
        for i in range(len(ids)-1, -1, -1):
            if distances[i] > distance_threshold:
                ids.pop(i)
                docs.pop(i)
                distances.pop(i)
    return query_result


In [18]:
filtered_claim_query_result = filter_query_result(claim_query_result)

现在使用这个更干净的上下文来评估论断。 

In [19]:
gpt_with_filtered_context_evaluation = assess_claims_with_context(claims, filtered_claim_query_result['documents'])
confusion_matrix(gpt_with_filtered_context_evaluation, groundtruth)

	Groundtruth
	True	False	NEE
True	9	0	1	
False	0	7	0	
NEE	8	5	20	


{'True': {'True': 9, 'False': 0, 'NEE': 1},
 'False': {'True': 0, 'False': 7, 'NEE': 0},
 'NEE': {'True': 8, 'False': 5, 'NEE': 20}}

## 结果


现在，在证据不足时，模型将论断评估为 True 或 False 的情况少了很多。然而，它也变得谨慎得多，倾向于将大多数条目标记为证据不足，偏离了确定性的判断。现在大多数论断都被评估为证据不足，因为其中很大一部分被距离阈值过滤掉了。可以通过调整距离阈值来找到最佳工作点，但这可能比较困难，而且取决于数据集和嵌入模型。 

# 假设性文档嵌入：有效利用幻觉

我们希望能够检索到相关文档，同时不检索那些可能干扰模型的不太相关的文档。实现这一目标的一种方法是改进检索查询。 

到目前为止，我们一直使用 _论断_（单句陈述）来查询数据集，而语料库包含的是描述科学论文的 _摘要_。直观地说，两者虽然可能相关，但在结构和含义上存在显著差异。这些差异会被嵌入模型编码，从而影响查询与最相关结果之间的距离。 

我们可以借助 LLM 生成相关文本的能力来克服这一问题。虽然其中的事实可能是幻觉，但模型生成的文档在内容和结构上比查询更接近语料库中的文档。这可能带来更好的查询，从而得到更好的结果。 

这种方法称为[假设性文档嵌入（HyDE）](https://arxiv.org/abs/2212.10496)，已被证明在检索任务上效果相当好。它应该能帮助我们把更多相关信息引入上下文，而不会污染上下文。  

简而言之：
- 嵌入完整摘要比嵌入单个句子能得到好得多的匹配
- 但论断通常是单个句子
- 因此 HyDE 表明，先用 GPT3 将论断扩展为幻觉生成的摘要，再基于这些摘要进行搜索（论断 -> 摘要 -> 结果），比直接搜索（论断 -> 结果）效果更好

首先，我们使用上下文示例提示模型，为每个要评估的论断生成与语料库中内容相似的文档。 

In [20]:
def build_hallucination_prompt(claim):
    return [{'role': 'system', 'content': """I will ask you to write an abstract for a scientific paper which supports or refutes a given claim. It should be written in scientific language, include a title. Output only one abstract, then stop.

    An Example:

    Claim:
    A high microerythrocyte count raises vulnerability to severe anemia in homozygous alpha (+)- thalassemia trait subjects.

    Abstract:
    BACKGROUND The heritable haemoglobinopathy alpha(+)-thalassaemia is caused by the reduced synthesis of alpha-globin chains that form part of normal adult haemoglobin (Hb). Individuals homozygous for alpha(+)-thalassaemia have microcytosis and an increased erythrocyte count. Alpha(+)-thalassaemia homozygosity confers considerable protection against severe malaria, including severe malarial anaemia (SMA) (Hb concentration < 50 g/l), but does not influence parasite count. We tested the hypothesis that the erythrocyte indices associated with alpha(+)-thalassaemia homozygosity provide a haematological benefit during acute malaria.
    METHODS AND FINDINGS Data from children living on the north coast of Papua New Guinea who had participated in a case-control study of the protection afforded by alpha(+)-thalassaemia against severe malaria were reanalysed to assess the genotype-specific reduction in erythrocyte count and Hb levels associated with acute malarial disease. We observed a reduction in median erythrocyte count of approximately 1.5 x 10(12)/l in all children with acute falciparum malaria relative to values in community children (p < 0.001). We developed a simple mathematical model of the linear relationship between Hb concentration and erythrocyte count. This model predicted that children homozygous for alpha(+)-thalassaemia lose less Hb than children of normal genotype for a reduction in erythrocyte count of >1.1 x 10(12)/l as a result of the reduced mean cell Hb in homozygous alpha(+)-thalassaemia. In addition, children homozygous for alpha(+)-thalassaemia require a 10% greater reduction in erythrocyte count than children of normal genotype (p = 0.02) for Hb concentration to fall to 50 g/l, the cutoff for SMA. We estimated that the haematological profile in children homozygous for alpha(+)-thalassaemia reduces the risk of SMA during acute malaria compared to children of normal genotype (relative risk 0.52; 95% confidence interval [CI] 0.24-1.12, p = 0.09).
    CONCLUSIONS The increased erythrocyte count and microcytosis in children homozygous for alpha(+)-thalassaemia may contribute substantially to their protection against SMA. A lower concentration of Hb per erythrocyte and a larger population of erythrocytes may be a biologically advantageous strategy against the significant reduction in erythrocyte count that occurs during acute infection with the malaria parasite Plasmodium falciparum. This haematological profile may reduce the risk of anaemia by other Plasmodium species, as well as other causes of anaemia. Other host polymorphisms that induce an increased erythrocyte count and microcytosis may confer a similar advantage.

    End of example.

    """}, {'role': 'user', 'content': f""""
    Perform the task for the following claim.

    Claim:
    {claim}

    Abstract:
    """}]


def hallucinate_evidence(claims):
    responses = []
    # Query the OpenAI API
    for claim in claims:
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=build_hallucination_prompt(claim),
        )
        responses.append(response.choices[0].message.content)
    return responses

我们为每个论断幻觉生成一篇文档。

*注意：这可能需要一些时间，100 个论断大约需要 7 分钟*。你可以减少要评估的论断数量，以更快地得到结果。 

In [21]:
hallucinated_evidence = hallucinate_evidence(claims)

我们将幻觉生成的文档作为查询在语料库中检索，并使用相同的距离阈值过滤结果。 

In [22]:
hallucinated_query_result = scifact_corpus_collection.query(query_texts=hallucinated_evidence, include=['documents', 'distances'], n_results=3)
filtered_hallucinated_query_result = filter_query_result(hallucinated_query_result)

然后让模型使用新的上下文评估这些论断。 

In [23]:
gpt_with_hallucinated_context_evaluation = assess_claims_with_context(claims, filtered_hallucinated_query_result['documents'])
confusion_matrix(gpt_with_hallucinated_context_evaluation, groundtruth)

	Groundtruth
	True	False	NEE
True	13	0	3	
False	1	10	1	
NEE	3	2	17	


{'True': {'True': 13, 'False': 0, 'NEE': 3},
 'False': {'True': 1, 'False': 10, 'NEE': 1},
 'NEE': {'True': 3, 'False': 2, 'NEE': 17}}

## 结果

将 HyDE 与简单的距离阈值相结合带来了显著的提升。模型不再偏向于将论断评估为 True，也不再偏向于判定证据不足。它也能更频繁地正确判断出何时证据不足。

# 结论

基于文档语料库为 LLM 提供上下文，是将 LLM 的通用推理能力和自然语言交互能力应用到你自己数据上的一种强大技术。然而，需要知道的是，朴素的查询和检索未必能产生最好的结果！归根结底，理解数据将帮助你充分发挥基于检索的问答方法的作用。